# Feature Engineering Deep Dive

This notebook provides a **deeper exploration** of the feature engineering layer:

- `FeatureEngineer` orchestrator — end-to-end feature generation
- Sentiment feature merging (news + social)
- Cross-modal sentiment features
- Feature schema versioning
- Rolling window analysis — how features evolve over time

**Prerequisites:** Ingested price data and (optionally) news/sentiment data.

In [1]:
import sys
sys.path.insert(0, "../src")

from equity_lake.features.engineering import FeatureEngineer
from equity_lake.features.pipeline import FeaturePipeline
from equity_lake.features.dag import clean_02, features_03, raw_01
from equity_lake.storage.duckdb import EquityDataDB
from equity_lake.core.paths import LAKE_DIR
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from datetime import date, timedelta

print("Feature engineering modules loaded")

Feature engineering modules loaded


## 1. FeatureEngineer — End-to-End Generation

The `FeatureEngineer` class orchestrates: data loading -> Hamilton DAG computation -> optional sentiment merge -> schema versioning.

In [2]:
engineer = FeatureEngineer()

print("FeatureEngineer initialized")
print(f"\nAvailable methods:")
for method in ["generate_features", "merge_sentiment_features", "merge_social_sentiment_features", "add_cross_modal_sentiment_features"]:
    print(f"  - {method}")

2026-06-09 09:26:07 [info     ] DuckDB views created successfully


FeatureEngineer initialized

Available methods:
  - generate_features
  - merge_sentiment_features
  - merge_social_sentiment_features
  - add_cross_modal_sentiment_features


In [3]:
# Generate features for a ticker
ticker = "AAPL"
end_date = date.today() - timedelta(days=1)
start_date = end_date - timedelta(days=180)

try:
    features_df = engineer.generate_features(
        tickers=[ticker],
        start_date=start_date,
        end_date=end_date,
        compute_target=True,
        include_sentiment=False,
    )
    print(f"Generated features: {features_df.shape}")
    print(f"Columns: {list(features_df.columns)[:20]}...")
    features_df.head()
except Exception as e:
    print(f"Feature generation error: {e}")
    print("\nMake sure price data exists in data/lake/")
    features_df = pd.DataFrame()

2026-06-09 09:26:07 [info     ] Generating features for 1 tickers from 2025-12-10 to 2026-06-08


2026-06-09 09:26:07 [debug    ] Executing parameterized query for 1 tickers


2026-06-09 09:26:08 [info     ] Loaded 124 rows of OHLCV data 


Computing features:   0%|          | 0/1 [00:00<?, ?it/s]

[volume:range_validator] validator failed. Message was: Series contains 0 values in range (0,None), and 124 outside.. Diagnostic information is: {'range': (0, None), 'in_range': 0, 'out_range': 124, 'data_size': 124}.


[rsi_14:range_validator] validator failed. Message was: Series contains 110 values in range (0,100), and 14 outside.. Diagnostic information is: {'range': (0, 100), 'in_range': 110, 'out_range': 14, 'data_size': 124}.


/Users/minghao/Desktop/personal/equity_lake/docs/notebooks/../../src/equity_lake/features/indicators.py:63: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return series.pct_change(length) * 100
/Users/minghao/Desktop/personal/equity_lake/docs/notebooks/../../src/equity_lake/features/hamilton_features.py:120: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return close.pct_change(1)
/Users/minghao/Desktop/personal/equity_lake/docs/notebooks/../../src/equity_lake/features/hamilton_features.py:124: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a

2026-06-09 09:26:08 [info     ] Generated 109 rows of features with 40 columns


2026-06-09 09:26:08 [debug    ] Feature columns: ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'bb_upper', 'bb_middle', 'bb_lower', 'bb_width', 'bb_pct', 'atr_14', 'roc_5', 'roc_10', 'roc_20', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'overnight_return', 'intraday_return', 'hl_range', 'volume_ma_20', 'volume_roc_5', 'obv', 'volume_ratio', 'day_of_week', 'day_of_month', 'month', 'quarter', 'days_to_month_end', 'trading_day_of_month', 'volatility_20', 'next_day_return', 'feature_schema_version']


Generated features: (109, 40)
Columns: ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'bb_upper', 'bb_middle', 'bb_lower', 'bb_width', 'bb_pct', 'atr_14', 'roc_5', 'roc_10', 'roc_20']...


## 2. Feature Schema Versioning

Every feature row is stamped with a `FEATURE_SCHEMA_VERSION` for reproducibility.

In [4]:
if not features_df.empty and "feature_schema_version" in features_df.columns:
    print(f"Schema version: {features_df['feature_schema_version'].unique()}")
    print(f"\nSchema version distribution:")
    print(features_df["feature_schema_version"].value_counts())
elif not features_df.empty:
    print("Schema version column not found — features may be from an older version")
    print(f"Available columns: {list(features_df.columns)}")
else:
    print("No features generated — skipping schema inspection")

Schema version: [2]

Schema version distribution:
feature_schema_version
2    109
Name: count, dtype: int64


## 3. Sentiment Feature Merging

Merge news and social sentiment data with price features.

In [5]:
if not features_df.empty:
    try:
        features_with_news = engineer.merge_sentiment_features(
            features_df=features_df,
            ticker=ticker,
        )
        sentiment_cols = [c for c in features_with_news.columns if "sentiment" in c.lower() or "news" in c.lower()]
        print(f"Features after news merge: {features_with_news.shape}")
        print(f"Sentiment columns added: {sentiment_cols}")
    except Exception as e:
        print(f"News sentiment merge: {e}")
        features_with_news = features_df
        sentiment_cols = []
else:
    features_with_news = pd.DataFrame()
    sentiment_cols = []

News sentiment merge: FeatureEngineer.merge_sentiment_features() got an unexpected keyword argument 'ticker'


In [6]:
# Merge social sentiment
if not features_with_news.empty:
    try:
        features_with_social = engineer.merge_social_sentiment_features(
            features_df=features_with_news,
            ticker=ticker,
        )
        social_cols = [c for c in features_with_social.columns if "social" in c.lower()]
        print(f"Features after social merge: {features_with_social.shape}")
        print(f"Social columns: {social_cols}")
    except Exception as e:
        print(f"Social sentiment merge: {e}")
        features_with_social = features_with_news
else:
    features_with_social = features_with_news

Social sentiment merge: FeatureEngineer.merge_social_sentiment_features() got an unexpected keyword argument 'ticker'


## 4. Cross-Modal Sentiment Features

Advanced features combining multiple sentiment sources.

In [7]:
if not features_with_social.empty:
    try:
        features_cross = engineer.add_cross_modal_sentiment_features(features_with_social)
        cross_cols = [c for c in features_cross.columns if "cross" in c.lower()]
        print(f"Cross-modal columns: {cross_cols}")
        if cross_cols:
            features_cross[cross_cols].describe()
    except Exception as e:
        print(f"Cross-modal features: {e}")

Cross-modal columns: []


## 5. Feature Evolution Over Time

Visualize how key features change over rolling windows.

In [8]:
if not features_df.empty:
    numeric_features = features_df.select_dtypes(include=[np.number]).columns.tolist()
    key_features = [f for f in ["rsi_14", "volatility_20", "bb_pct", "volume_ratio", "atr_14"] if f in numeric_features]

    if key_features:
        fig = make_subplots(
            rows=len(key_features), cols=1,
            shared_xaxes=True,
            subplot_titles=key_features,
            vertical_spacing=0.05,
        )

        date_col = "date" if "date" in features_df.columns else None
        x = features_df[date_col] if date_col else features_df.index

        for i, feat in enumerate(key_features):
            fig.add_trace(go.Scatter(
                x=x, y=features_df[feat], name=feat, line=dict(width=1),
            ), row=i + 1, col=1)

        fig.update_layout(title=f"Feature Evolution — {ticker}", height=200 * len(key_features), showlegend=False)
        fig.show()
    else:
        print(f"Key features not found. Available: {numeric_features[:10]}")
else:
    print("No features to visualize")

## 6. Rolling Statistics

Compute rolling mean and standard deviation for key features.

In [9]:
if not features_df.empty:
    target_feature = None
    for f in ["rsi_14", "volatility_20", "atr_14"]:
        if f in features_df.columns:
            target_feature = f
            break

    if target_feature:
        date_col = "date" if "date" in features_df.columns else None
        x = features_df[date_col] if date_col else features_df.index

        rolling_mean = features_df[target_feature].rolling(20).mean()
        rolling_std = features_df[target_feature].rolling(20).std()

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=x, y=features_df[target_feature], name=target_feature, line=dict(width=1, color="blue")))
        fig.add_trace(go.Scatter(x=x, y=rolling_mean, name="20D Mean", line=dict(dash="dash", color="red")))
        fig.add_trace(go.Scatter(
            x=pd.concat([x, x.iloc[::-1]]),
            y=pd.concat([rolling_mean + 2 * rolling_std, (rolling_mean - 2 * rolling_std).iloc[::-1]]),
            fill="toself", fillcolor="rgba(255,0,0,0.1)",
            line=dict(color="rgba(255,255,255,0)"), name="±2σ Band",
        ))

        fig.update_layout(title=f"{target_feature} — Rolling Mean & Bands", height=500)
        fig.show()
else:
    print("No data for rolling analysis")

## 7. Feature Distribution Analysis

Examine the distribution of each feature.

In [10]:
if not features_df.empty:
    numeric_features = features_df.select_dtypes(include=[np.number]).columns.tolist()
    plot_features = [f for f in ["rsi_14", "volatility_20", "bb_pct", "return_1d", "return_5d"] if f in numeric_features]

    if plot_features:
        fig = make_subplots(rows=1, cols=len(plot_features), subplot_titles=plot_features)
        for i, feat in enumerate(plot_features):
            fig.add_trace(go.Histogram(x=features_df[feat], name=feat, nbinsx=40), row=1, col=i + 1)
        fig.update_layout(title="Feature Distributions", height=400, showlegend=False)
        fig.show()
else:
    print("No features for distribution analysis")

## Next Steps

- **[06-ml-prediction.ipynb](06-ml-prediction.ipynb)** — Use features for XGBoost prediction
- **[07-signal-scanning.ipynb](07-signal-scanning.ipynb)** — Generate signals from features + ML
- **[09-sentiment-analysis.ipynb](09-sentiment-analysis.ipynb)** — Deep dive into sentiment scoring